# Load datasets into the bronze landing volume

This notebook copies the local synthetic datasets into the governed Bronze landing volume using the expected directory structure:

- `claims_1000.csv` -> `/Volumes/<catalog>/<schema>/<volume>/claims/`
- `providers_1000.csv` -> `/Volumes/<catalog>/<schema>/<volume>/providers/`
- `diagnosis.csv` -> `/Volumes/<catalog>/<schema>/<volume>/diagnosis/`
- `cost.csv` -> `/Volumes/<catalog>/<schema>/<volume>/cost/`
- `datasets/policies/*.pdf` -> `/Volumes/<catalog>/<schema>/<volume>/policies/`

Existing files are overwritten.


In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import sys


def _repo_root() -> Path:
    search_roots: list[Path] = []

    if "__file__" in globals():
        search_roots.append(Path(__file__).resolve().parent)

    search_roots.append(Path.cwd().resolve())

    seen: set[Path] = set()
    for root in search_roots:
        for candidate in (root, *root.parents):
            if candidate in seen:
                continue
            seen.add(candidate)
            if (candidate / "datasets" / "claims_1000.csv").exists() and (
                candidate / "src" / "common" / "bronze_sources.py"
            ).exists():
                return candidate

    raise ModuleNotFoundError(
        "Could not locate the synced repo root containing datasets/ and src/common/bronze_sources.py."
    )


REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.common.bronze_sources import BRONZE_SOURCES, POLICY_SOURCE


def _ensure_directory(path: str) -> None:
    dbutils.fs.mkdirs(path)


def _copy_overwrite(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        destination.unlink()
    shutil.copy2(source, destination)


def _path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


In [ ]:
dbutils.widgets.text("catalog", "healthcare", "Catalog")
dbutils.widgets.text("schema", "bronze", "Schema")
dbutils.widgets.text("volume", "raw_landing", "Volume")
dbutils.widgets.text("datasets_root", str((REPO_ROOT / "datasets").resolve()), "Datasets root")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
VOLUME = dbutils.widgets.get("volume").strip()
DATASETS_ROOT = Path(dbutils.widgets.get("datasets_root").strip()).expanduser().resolve()

if not CATALOG or not SCHEMA or not VOLUME:
    raise ValueError("Catalog, schema, and volume widget values are required.")

if not DATASETS_ROOT.exists():
    raise FileNotFoundError(f"Datasets root does not exist: {DATASETS_ROOT}")

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
if not _path_exists(VOLUME_ROOT):
    raise FileNotFoundError(
        f"Landing volume root does not exist: {VOLUME_ROOT}. Run the bootstrap notebook first."
    )

print(f"Repo datasets root: {DATASETS_ROOT}")
print(f"Landing volume root: {VOLUME_ROOT}")


In [ ]:
manifest_rows: list[dict[str, str]] = []

for dataset_key, source in BRONZE_SOURCES.items():
    source_file = DATASETS_ROOT / source.local_filename
    target_dir = Path(VOLUME_ROOT) / source.volume_subdirectory
    target_file = target_dir / source.local_filename

    if not source_file.exists():
        raise FileNotFoundError(f"Missing dataset file: {source_file}")

    _ensure_directory(str(target_dir))
    _copy_overwrite(source_file, target_file)

    manifest_rows.append(
        {
            "dataset": dataset_key,
            "source": str(source_file),
            "target": str(target_file),
            "status": "copied",
        }
    )

policies_source_dir = DATASETS_ROOT / POLICY_SOURCE.volume_subdirectory
policies_target_dir = Path(VOLUME_ROOT) / POLICY_SOURCE.volume_subdirectory

if not policies_source_dir.exists():
    raise FileNotFoundError(f"Missing policies directory: {policies_source_dir}")

pdf_sources = sorted(policies_source_dir.glob("*.pdf"))
if not pdf_sources:
    raise FileNotFoundError(f"No PDF files found in {policies_source_dir}")

_ensure_directory(str(policies_target_dir))

for pdf_file in pdf_sources:
    target_file = policies_target_dir / pdf_file.name
    _copy_overwrite(pdf_file, target_file)
    manifest_rows.append(
        {
            "dataset": "policies",
            "source": str(pdf_file),
            "target": str(target_file),
            "status": "copied",
        }
    )

for row in manifest_rows:
    print(f"{row['status']}: {row['source']} -> {row['target']}")


In [ ]:
verification_rows: list[dict[str, str]] = []
missing_targets: list[str] = []

for dataset_key, source in BRONZE_SOURCES.items():
    target_file = Path(VOLUME_ROOT) / source.volume_subdirectory / source.local_filename
    exists = target_file.exists()
    verification_rows.append(
        {
            "dataset": dataset_key,
            "target": str(target_file),
            "status": "present" if exists else "missing",
        }
    )
    if not exists:
        missing_targets.append(str(target_file))

for pdf_file in pdf_sources:
    target_file = policies_target_dir / pdf_file.name
    exists = target_file.exists()
    verification_rows.append(
        {
            "dataset": "policies",
            "target": str(target_file),
            "status": "present" if exists else "missing",
        }
    )
    if not exists:
        missing_targets.append(str(target_file))

for row in verification_rows:
    print(f"{row['status']}: {row['target']}")

if missing_targets:
    raise AssertionError("Some files were not copied: " + ", ".join(missing_targets))

print("All datasets copied into the bronze landing volume with the expected structure.")
